In [ ]:
!pip install -U dspy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.3/297.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.9/395.9 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 15.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does n

In [ ]:
import dspy
import json

In [ ]:
# The OPenAI API key
key = ''

In [ ]:
lm = dspy.LM('openai/gpt-4o-mini', api_key=key)
dspy.configure(lm=lm)

In [ ]:
from typing import Literal

class Classify(dspy.Signature):
    """
    Classify the presence of stigmatizing content in a social media post related to diabetes.
    The output should indicate whether the post includes stigma, does not include stigma, or is unclear.
    """

    post: str = dspy.InputField(desc="A social media post that may or may not express stigma toward people with diabetes.")

    category: Literal['Yes Stigma', 'No Stigma', 'Unclear'] = dspy.OutputField(
        desc="Label indicating whether the post contains stigmatizing content toward people with diabetes."
    )
    confidence: float = dspy.OutputField(desc="Model's confidence in the classification decision.")

classify = dspy.Predict(Classify)
classify(post='When I first got diagnosed at 16, I craved validation for the hurt I was feeling. There was none. I just saw article after article saying I could “Live a normal life”. I would write letters to myself about how angry I was and feel like an idiot. I started drinking by myself. I wish this had been here when I was 16, I’m so grateful it is for the next generation.')

Prediction(
    category='No Stigma',
    confidence=0.85
)

In [ ]:
from typing import Literal, List
import dspy

class ClassifyDiabetesStigma(dspy.Signature):
    """
    Classify the presence and type(s) of stigma in a social media post related to diabetes.

    You are given the title and text of a social media post. Your tasks are:

    1. Classify whether the post expresses stigma toward people with diabetes:
       - Yes Stigma: Contains stigmatizing content or implications.
       - No Stigma: Neutral, supportive, or stigma-free.
       - Unclear: Ambiguous or insufficient information.

    2. If the post contains stigma (Yes Stigma), identify all applicable types:
       - Experienced stigma: Direct discrimination or exclusion due to diabetes.
       - Perceived stigma: Awareness or assumption that others hold negative views.
       - Anticipated stigma: Fear or expectation of being stigmatized.
       - Internalized stigma: Shame or self-blame from negative diabetes stereotypes.
       - Intersectional stigma: Stigma compounded by other factors (e.g., obesity, race).

       If there is no stigma (No Stigma or Unclear), return an empty list for stigma_type.
    """

    title: str = dspy.InputField(desc="The title of the social media post.")
    post: str = dspy.InputField(desc="The main text or content of the social media post.")

    category: Literal['Yes Stigma', 'No Stigma', 'Unclear'] = dspy.OutputField(
        desc="Whether the post expresses diabetes-related stigma."
    )
    stigma_type: List[Literal['experienced stigma', 'perceived stigma', 'anticipated stigma', 'internalized stigma', 'intersectional stigma']] = dspy.OutputField(
        desc="List of stigma types expressed in the post, or an empty list if not stigmatizing."
    )
    confidence: float = dspy.OutputField(desc="Model's confidence in its classification.")


In [ ]:
classify_stigma = dspy.Predict(ClassifyDiabetesStigma)
classify_stigma(title = 'I don\'t know what to do', post='When I first got diagnosed at 16, I craved validation for the hurt I was feeling. There was none. I just saw article after article saying I could “Live a normal life”. I would write letters to myself about how angry I was and feel like an idiot. I started drinking by myself. I wish this had been here when I was 16, I’m so grateful it is for the next generation.')

Prediction(
    category='Yes Stigma',
    stigma_type=['internalized stigma'],
    confidence=0.85
)

In [ ]:
with open('/content/drive/MyDrive/stigma_project/data/all_data.json', 'r') as f:
  data = json.loads(f.read())
  f.close()

In [ ]:
data[0]

{'title': 'Weekly r/diabetes vent thread',
 'post text': "Tell us the crap you're dealing with this week. Did someone suggest cinnamon again? What about that relative who tried to pray the beetus away?\n  \n As always, please keep in mind [our rules](https://www.reddit.com/r/diabetes/about/rules)",
 'label': 'Yes Stigma',
 'stigma_types': ['perceived stigma']}

In [ ]:
trainset = []

for post_obj in data:
    example = dspy.Example(title = post_obj['title'], post=post_obj['post text'], category=post_obj['label'], stigma_type=post_obj['stigma_types']).with_inputs("title", "post")
    trainset.append(example)

In [ ]:
from dspy.teleprompt import *

# Load our model
lm = dspy.LM('openai/gpt-4o', api_key=key)
prompt_gen_lm = dspy.LM('openai/gpt-4o-mini', api_key=key)

dspy.configure(lm=lm)

In [ ]:
def validate_category(example, prediction, trace=None):
    return (prediction.category == example.category) and sorted(prediction.stigma_type) == sorted(example.stigma_type)

In [ ]:
# Optimize
tp = dspy.MIPROv2(metric=validate_category, auto="light", prompt_model=prompt_gen_lm, task_model=lm)
optimized_classify = tp.compile(classify_stigma, trainset=trainset, max_labeled_demos=0, max_bootstrapped_demos=0, requires_permission_to_run=False)

2025/06/30 11:39:52 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 10
minibatch: True
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 100

2025/06/30 11:39:52 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/06/30 11:39:52 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used for informing instruction proposal.

2025/06/30 11:39:52 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6


  1%|▏         | 1/68 [00:01<01:15,  1.13s/it]


Bootstrapped 1 full traces after 1 examples for up to 1 rounds, amounting to 1 attempts.
Bootstrapping set 3/6


 10%|█         | 7/68 [00:08<01:15,  1.23s/it]


...

Average Metric: 82.00 / 89 (92.1%):  88%|████████▊ | 88/100 [00:07<00:00, 14.09it/s]

2025/06/30 11:41:55 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Average Metric: 90.00 / 100 (90.0%): 100%|██████████| 100/100 [00:08<00:00, 11.66it/s]

2025/06/30 11:41:56 INFO dspy.evaluate.evaluate: Average Metric: 90 / 100 (90.0%)
2025/06/30 11:41:56 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [90.0, 90.0, 90.0]
2025/06/30 11:41:56 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 90.0


2025/06/30 11:41:56 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2025/06/30 11:41:56 INFO dspy.teleprompt.mipro_optimizer_v2: 

2025/06/30 11:41:56 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 90.0!


In [ ]:
optimized_classify

Predict(ClassifyDiabetesStigma(title, post -> category, stigma_type, confidence
    instructions='Classify the presence and type(s) of stigma in a social media post related to diabetes.\n\nYou are given the title and text of a social media post. Your tasks are:\n\n1. Classify whether the post expresses stigma toward people with diabetes:\n   - Yes Stigma: Contains stigmatizing content or implications.\n   - No Stigma: Neutral, supportive, or stigma-free.\n   - Unclear: Ambiguous or insufficient information.\n\n2. If the post contains stigma (Yes Stigma), identify all applicable types:\n   - Experienced stigma: Direct discrimination or exclusion due to diabetes.\n   - Perceived stigma: Awareness or assumption that others hold negative views.\n   - Anticipated stigma: Fear or expectation of being stigmatized.\n   - Internalized stigma: Shame or self-blame from negative diabetes stereotypes.\n   - Intersectional stigma: Stigma compounded by other factors (e.g., obesity, race).\n   \n   If

In [ ]:
optimized_classify.save('drive/MyDrive/multi_class_optimized_program.json')

In [ ]:
classify

Predict(Classify(post -> category, confidence
    instructions='Classify the presence of stigmatizing content in a social media post related to diabetes.\nThe output should indicate whether the post includes stigma, does not include stigma, or is unclear.'
    post = Field(annotation=str required=True json_schema_extra={'desc': 'A social media post that may or may not express stigma toward people with diabetes.', '__dspy_field_type': 'input', 'prefix': 'Post:'})
    category = Field(annotation=Literal['Yes Stigma', 'No Stigma', 'Unclear'] required=True json_schema_extra={'desc': 'Label indicating whether the post contains stigmatizing content toward people with diabetes.', '__dspy_field_type': 'output', 'prefix': 'Category:'})
    confidence = Field(annotation=float required=True json_schema_extra={'desc': "Model's confidence in the classification decision.", '__dspy_field_type': 'output', 'prefix': 'Confidence:'})
))

In [ ]:
new_classify = dspy.Predict(Classify)

In [ ]:
optimized_classify(post='When I first got diagnosed at 16, I craved validation for the hurt I was feeling. There was none. I just saw article after article saying I could “Live a normal life”. I would write letters to myself about how angry I was and feel like an idiot. I started drinking by myself. I wish this had been here when I was 16, I’m so grateful it is for the next generation.')

Prediction(
    category='No Stigma',
    confidence=0.85
)

In [ ]:
new_classify = dspy.Predict(Classify)
new_classify.load('drive/MyDrive/optimized_program.json')

new_classify

Predict(StringSignature(post -> category, confidence
    instructions="In a high-stakes situation where someone is feeling overwhelmed by their diabetes management and is seeking support online, classify the presence of stigmatizing content in their social media post. Determine whether the post includes stigma, does not include stigma, or if the content is unclear. Your response should reflect the emotional weight of the user's experience and the potential impact of stigma on their mental health."
    post = Field(annotation=str required=True json_schema_extra={'desc': 'A social media post that may or may not express stigma toward people with diabetes.', '__dspy_field_type': 'input', 'prefix': 'Post:'})
    category = Field(annotation=Literal['Yes Stigma', 'No Stigma', 'Unclear'] required=True json_schema_extra={'desc': 'Label indicating whether the post contains stigmatizing content toward people with diabetes.', '__dspy_field_type': 'output', 'prefix': 'Category:'})
    confidence = 

In [ ]:
optimized_classify(title = 'I don\'t know what to do', post='When I first got diagnosed at 16, I craved validation for the hurt I was feeling. There was none. I just saw article after article saying I could “Live a normal life”. I would write letters to myself about how angry I was and feel like an idiot. I started drinking by myself. I wish this had been here when I was 16, I’m so grateful it is for the next generation.')

Prediction(
    category='Yes Stigma',
    stigma_type=['internalized stigma'],
    confidence=0.85
)

# Starting the expansion with the classifier

In [ ]:
with open('/content/drive/MyDrive/stigma_project/data/filtered_val_data.json', 'r') as f:
  val_data = json.loads(f.read())
  f.close()

In [ ]:
val_data[0]

{'title': 'Diabetic-friendly pancakes?',
 'post text': "My friend was diagnosed with prediabetes and now has to change her diet. For breakfast, he would eat pancakes or waffles along with fruit and orange juice, the reason being he really likes them. He tends to have a big breakfast/lunch to keep him full, will skip lunch, and have dinner. This will be a struggle for him, and I don't think he wants to give up the breakfast. So, are there any diabetic-friendly pancakes and waffles he could have every day? Isn't there some kind of healthy substitute that won't affect his insulin or blood sugar?"}

In [ ]:
b = optimized_classify(title=val_data[0]['title'], post=val_data[0]['post text'])

In [ ]:
b.stigma_type

[]

In [ ]:
preds = []

for i in range(800):
  pred = optimized_classify(title=val_data[i]['title'], post=val_data[i]['post text'])
  val_data[i]['category'] = pred.category
  val_data[i]['stigma_types'] = pred.stigma_type
  preds.append(val_data[i])

{'title': 'Blood Sugar On Work Days vs Weekends',
 'post text': 'Hello All!\n\nOut of curiosity, has anyone noticed that their blood sugar runs higher on days when they are at work vs when they’re off work?\n\nEvery day, wake up at the same time. Fasting blood sugar is right around 98-110. Eat the same exact thing for breakfast everyday and take the same exact insulin dosages (40u Long, 35u Fast) for 20g of carbs. On a day off, my numbers go up to around 105 and then stay pretty level around 98-100. On a work day, two hours later, my sugar shoots up to 150-160. I have to take an additional 15u to get it to come back down to 98-100 range. \n\nDoes anyone else notice this? How do you manage this? My in-person days are worse - I’ve had it jump up to 180-190 if I don’t have time to take another fast acting dose before walking in the door at work two hours-post injection. \n\nThanks!',
 'category': 'No Stigma',
 'stigma_types': []}

In [ ]:
with open('/content/drive/MyDrive/stigma_project/data/800_labeled_val_data.json', 'w') as f:
  f.write(json.dumps(preds))
  f.close()

In [ ]:
for p in preds:
  if p['category'] == 'Yes Stigma':
    print(p['stigma_types'])

['internalized stigma']
['internalized stigma']
['experienced stigma', 'perceived stigma', 'anticipated stigma']
['experienced stigma', 'perceived stigma', 'anticipated stigma']
['internalized stigma', 'anticipated stigma']
['internalized stigma']
['experienced stigma']
['experienced stigma', 'perceived stigma']
['internalized stigma']
['experienced stigma', 'perceived stigma', 'internalized stigma']
['experienced stigma', 'perceived stigma']
['internalized stigma']
['experienced stigma', 'perceived stigma']
['experienced stigma']
['internalized stigma']
['anticipated stigma']
['experienced stigma', 'perceived stigma']
['experienced stigma', 'internalized stigma', 'intersectional stigma']
['internalized stigma']
['experienced stigma', 'perceived stigma']
['experienced stigma', 'perceived stigma']
['experienced stigma', 'perceived stigma']
['experienced stigma', 'perceived stigma']
['internalized stigma']
['experienced stigma', 'perceived stigma']
['perceived stigma', 'anticipated stigm